# Phase 1: formal witness pipeline regression

Ten synthetic Golden/mutant pairs with known distinguishing inputs. Gate: >=90% SAT-to-replay on supported cases, funnel accounting for 100% of calls. No benchmark data involved.


In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile, shutil
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
m=list(INPUT.rglob('active_tb_witness_v1/PHASE1_BOUNDARY.json'))
if len(m)==1: SRC=m[0].parent
else:
 a=list(INPUT.rglob('active_tb_witness_v1.zip')); assert len(a)==1,a
 d=WORK/'attached'
 with zipfile.ZipFile(a[0]) as zf: zf.extractall(d)
 SRC=d/'active_tb_witness_v1'
print(json.loads((SRC/'PHASE1_BOUNDARY.json').read_text()))
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','yosys','iverilog','z3'],check=True)
for t in ('yosys','yosys-smtbmc','iverilog','vvp','z3'):
    p=subprocess.run(['which',t],capture_output=True,text=True)
    print(f'{t:14s}', p.stdout.strip() or 'MISSING')
subprocess.run(['yosys','-V'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','pyyaml','click'],check=False)


In [ ]:
OUT=WORK/'witness_out'; OUT.mkdir(exist_ok=True)
r=subprocess.run([sys.executable,str(SRC/'witness_regression.py'),
                  '--bounds','4,8,16,32','--timeout','90',
                  '--out',str(OUT/'witness_funnel.csv')],
                 text=True,capture_output=True,cwd=SRC)
print(r.stdout[-6000:])
if r.returncode: print('STDERR:', r.stderr[-3000:])
shutil.make_archive(str(WORK/'witness_out'),'zip',OUT)
print('packaged')
